# 05 · Race results from Jolpica (2018–2025)

Finishing order, grid position, status (finished / lapped / retirement reason) and points for every race, from the free [Jolpica](https://github.com/jolpica/jolpica-f1) API (the successor to Ergast; no login needed). About five requests per season, so one run covers everything.

These are the targets and key inputs for the baseline finishing-position model. Run notebook 03 first so the cross-check against lap data works.

In [1]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/khushi075/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

try:
    import imp
except ImportError:
    import types, importlib
    imp = types.ModuleType('imp')
    imp.reload = importlib.reload
    sys.modules['imp'] = imp

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

Mounted at /content/drive
Cloning into '/content/f1-race-sim'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 121 (delta 66), reused 75 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 77.35 KiB | 2.50 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/f1-race-sim
Code: /content/f1-race-sim | Data: /content/drive/MyDrive/f1sim_data


In [2]:
# @title Download (one run is enough)
START_YEAR = 2018  # @param {type:"integer"}
END_YEAR = 2025  # @param {type:"integer"}
FORCE = False  # @param {type:"boolean"}
# FORCE re-downloads seasons already on Drive (e.g. to add circuit information)

from f1sim.paths import get_paths
from f1sim.ingest_results import ingest_results_seasons

paths = get_paths()
summary = ingest_results_seasons(range(START_YEAR, END_YEAR + 1), paths, force=FORCE)

saved  2018: 21 races, 420 rows
saved  2019: 21 races, 420 rows
saved  2020: 17 races, 340 rows
saved  2021: 22 races, 440 rows
saved  2022: 22 races, 440 rows
saved  2023: 22 races, 440 rows
saved  2024: 24 races, 479 rows
saved  2025: 24 races, 479 rows
Saved this run:     8
Already on disk:    0
Skipped (future):   0
Failed:             0


In [3]:
# @title Coverage per season
from f1sim.ingest_results import load_results

results = load_results(paths)
per_race = results.groupby(["Year", "Round"]).agg(
    Drivers=("DriverId", "size"),
    Classified=("ClassifiedPosition", lambda s: int(s.str.isdigit().sum())),
    PitLaneStarts=("GridPosition", lambda s: int((s == 0).sum())),
).reset_index()
per_race["NotClassified"] = per_race["Drivers"] - per_race["Classified"]
per_race.groupby("Year").agg(
    Races=("Round", "size"), MinDrivers=("Drivers", "min"), MaxDrivers=("Drivers", "max"),
    NotClassified=("NotClassified", "sum"), PitLaneStarts=("PitLaneStarts", "sum"),
)

,Races,MinDrivers,MaxDrivers,NotClassified,PitLaneStarts
Year,,,,,
2018,21,20,20,80,1
2019,21,20,20,50,12
2020,17,20,20,53,5
2021,22,20,20,52,12
2022,22,20,20,64,11
2023,22,20,20,50,3
2024,24,19,20,47,1
2025,24,19,20,55,0


In [4]:
# @title Cross-check: do results and lap data cover the same races, with the same names?
import pandas as pd
from f1sim.ingest_tracing import load_race_ti

lap_races = load_race_ti("laps", paths)[["Year", "Round", "Event"]].drop_duplicates()
res_races = results[["Year", "Round", "Event"]].drop_duplicates()
both = lap_races.merge(res_races, on=["Year", "Round"], how="outer",
                       suffixes=("Laps", "Results"), indicator=True)

only_one = both[both["_merge"] != "both"]
name_diff = both[(both["_merge"] == "both") & (both["EventLaps"] != both["EventResults"])]
print(f"Races in both: {(both['_merge'] == 'both').sum()} | only in one source: {len(only_one)} | "
      f"same round but different name: {len(name_diff)}")
pd.concat([only_one, name_diff])

Races in both: 173 | only in one source: 0 | same round but different name: 0


,Year,Round,EventLaps,EventResults,_merge


In [5]:
# @title Most common statuses (finished, lapped, retirement reasons)
results["Status"].value_counts().head(25)

,count
Status,
Finished,1995
+1 Lap,536
Lapped,297
Retired,161
+2 Laps,101
Collision,77
Accident,35
Engine,31
Collision damage,29
